In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup as bs
import pandas as pd

# webdriver_manager를 사용하여 Chrome 드라이버 설정
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

# 검색할 구 목록
districts = ["경기도", "강원도", "경상북도", "경상남도", 
             "충청북도", "충청남도", "전라남도","전라북도",
             "제주도", "서울특별시", "부산광역시", "대구광역시",
             "인천광역시", "광주광역시", "대전광역시", "울산광역시",
             "세종특별자치시"
]


# 결과를 저장할 리스트 생성
data = []

for district in districts:
    # 카카오 지도 웹사이트 열기
    driver.get("https://map.kakao.com/")
    
    # 요소가 나타날 때까지 기다린 후 클릭
    time.sleep(2)
    try:
        img = driver.find_element(By.CSS_SELECTOR, "div.inner_coach_layer")
        img.click()
    except:
        pass
    
    # 검색 수행
    search = driver.find_element(By.CSS_SELECTOR, "#search\\.keyword\\.query")
    search.send_keys(f"기차역 {district}")
    search_btn = driver.find_element(By.CSS_SELECTOR, "#search\\.keyword\\.submit")
    search_btn.click()
    
    # 검색 결과가 로드될 시간을 줌
    time.sleep(2)
    
    # BeautifulSoup을 사용하여 페이지 소스 파싱
    soup = bs(driver.page_source, "lxml")
    
    # 광고 제거
    ads = soup.select("span.ico_ad")
    for ad in ads:
        ad.extract()
    
    # "장소 더보기" 버튼 클릭
    try:
        btn_more = driver.find_element(By.CSS_SELECTOR, "#info\\.search\\.place\\.more")
        btn_more.click()
        time.sleep(2)
    except:
        pass
    
    try:
        while True:
            # 1부터 5까지 페이지를 반복
            for page_num in range(1, 6):
                try:
                    page_button = driver.find_element(By.CSS_SELECTOR, f"#info\\.search\\.page\\.no{page_num}")
                    page_button.click()
                    time.sleep(2)
                    
                    soup = bs(driver.page_source, "lxml")
                    
                    # 광고 제거
                    ads = soup.select("span.ico_ad")
                    for ad in ads:
                        ad.extract()
                    
                    # 장소 이름과 도로명 주소 추출
                    titles = soup.select("a.link_name")
                    addresses = soup.select("p[data-id='address']")
                    
                    for title, address in zip(titles, addresses):
                        data.append({
                            "장소": title['title'],
                            "도로명 주소": address['title'],
                            "지역": district
                        })
                except:
                    break
            
            # 다음 버튼 클릭
            next_button = driver.find_element(By.CSS_SELECTOR, "#info\\.search\\.page\\.next")
            if "disabled" in next_button.get_attribute("class"):
                break
            next_button.click()
            time.sleep(2)
    except Exception as e:
        print(f"오류 발생: {e}")

# 드라이버 닫기
driver.quit()

# 데이터를 DataFrame으로 변환
df = pd.DataFrame(data)

# DataFrame을 CSV 파일로 저장
df.to_csv("train_locations.csv", index=False, encoding='utf-8-sig')

print("데이터가 'train_locations.csv' 파일에 저장되었습니다.")


In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
import time

# CSV 파일 경로
input_file_path = r'C:\Users\kfq\Downloads\Final_Project\카카오맵 크롤링\train_locations.csv'
output_file_path = r'C:\Users\kfq\Downloads\Final_Project\카카오맵 크롤링\train_locations_geocode.csv'

# CSV 파일 읽기
try:
    df = pd.read_csv(input_file_path, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(input_file_path, encoding='cp949')

# 지오코더 초기화
geolocator = Nominatim(user_agent="socar_geocoder", timeout=10)  # timeout 값을 10초로 설정

# 지오코딩 함수 정의
def geocode_address(address, retries=5):
    for attempt in range(retries):
        try:
            location = geolocator.geocode(address)
            if location:
                return location.latitude, location.longitude
        except GeocoderTimedOut:
            print(f"Geocoding timed out on attempt {attempt + 1}. Retrying...")
            time.sleep(2)  # 2초 대기 후 재시도
        except GeocoderServiceError as e:
            print(f"Geocoding service error: {e}")
        except Exception as e:
            print(f"Unexpected error: {e}")
    return None, None  # 재시도 후에도 실패하면 None 반환

# 위도와 경도 컬럼 추가
latitudes = []
longitudes = []

for index, row in df.iterrows():
    lat, lon = None, None
    if pd.notna(row['장소']):
        # 쏘카존 지점으로 지오코딩 시도
        lat, lon = geocode_address(row['장소'])
    if (lat is None or lon is None) and pd.notna(row['도로명 주소']):
        # 실패하면 도로명 주소로 지오코딩 시도
        lat, lon = geocode_address(row['도로명 주소'])
    
    latitudes.append(lat)
    longitudes.append(lon)

df['위도'] = latitudes
df['경도'] = longitudes

# 위도와 경도만 포함된 새로운 데이터프레임 생성
df_geocoded = df[['위도', '경도']]

# 결과를 새로운 CSV 파일로 저장 (utf-8 인코딩)
df_geocoded.to_csv(output_file_path, index=False, encoding='utf-8')

print("지오코딩이 완료되었습니다. 결과가 저장되었습니다:", output_file_path)


In [ ]:
import pandas as pd

# 데이터 로드
df = pd.read_csv("train_locations.csv", encoding='utf-8-sig')

# "(폐역)"이 포함된 장소를 필터링하여 제거
df_filtered = df[~df['장소'].str.contains('\(폐역\)', na=False)]

# 필터링된 데이터를 새로운 CSV 파일로 저장
df_filtered.to_csv("filtered_train_locations.csv", index=False, encoding='utf-8-sig')

print("필터링된 데이터가 'filtered_train_locations.csv' 파일에 저장되었습니다.")


In [ ]:
import pandas as pd

# 데이터 로드
df = pd.read_csv("filtered_train_locations.csv", encoding='utf-8-sig')

# "(폐역)"이 포함된 장소를 필터링하여 제거
df_filtered = df[~df['장소'].str.contains('\(', na=False)]

# 필터링된 데이터를 새로운 CSV 파일로 저장
df_filtered.to_csv("filtered_train_locations2.csv", index=False, encoding='utf-8-sig')

print("필터링된 데이터가 'filtered_train_locations2.csv' 파일에 저장되었습니다.")


In [ ]:
import pandas as pd

# 데이터 로드
df = pd.read_csv("filtered_train_locations2.csv", encoding='utf-8-sig')

# "(폐역)"이 포함된 장소를 필터링하여 제거
df_filtered = df[~df['장소'].str.contains('신호장', na=False)]

# 필터링된 데이터를 새로운 CSV 파일로 저장
df_filtered.to_csv("filtered_train_locations3.csv", index=False, encoding='utf-8-sig')

print("필터링된 데이터가 'filtered_train_locations3.csv' 파일에 저장되었습니다.")


In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup as bs
import pandas as pd

# webdriver_manager를 사용하여 Chrome 드라이버 설정
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

# 검색할 구 목록
districts = ["서울특별시", "부산광역시", "대구광역시",
             "인천광역시", "광주광역시", "대전광역시", "울산광역시",
             "세종특별자치시"]

# 결과를 저장할 리스트 생성
data = []

for district in districts:
    # 카카오 지도 웹사이트 열기
    driver.get("https://map.kakao.com/")
    
    # 요소가 나타날 때까지 기다린 후 클릭
    time.sleep(2)
    try:
        img = driver.find_element(By.CSS_SELECTOR, "div.inner_coach_layer")
        img.click()
    except:
        pass
    
    # 검색 수행
    search = driver.find_element(By.CSS_SELECTOR, "#search\\.keyword\\.query")
    search.send_keys(f"기차역 {district}")
    search_btn = driver.find_element(By.CSS_SELECTOR, "#search\\.keyword\\.submit")
    search_btn.click()
    
    # 검색 결과가 로드될 시간을 줌
    time.sleep(2)
    
    # BeautifulSoup을 사용하여 페이지 소스 파싱
    soup = bs(driver.page_source, "lxml")
    
    # 광고 제거
    ads = soup.select("span.ico_ad")
    for ad in ads:
        ad.extract()
    
    # "장소 더보기" 버튼 클릭
    try:
        btn_more = driver.find_element(By.CSS_SELECTOR, "#info\\.search\\.place\\.more")
        btn_more.click()
        time.sleep(2)
    except:
        pass
    
    try:
        while True:
            # 페이지 번호가 있는지 확인
            try:
                page_buttons = driver.find_elements(By.CSS_SELECTOR, "#info\\.search\\.page\\.no")
                if not page_buttons:
                    # 페이지 번호가 없으면 데이터 추출
                    titles = driver.find_elements(By.CSS_SELECTOR, "a.link_name")
                    addresses = driver.find_elements(By.CSS_SELECTOR, "p[data-id='address']")
                    
                    for title, address in zip(titles, addresses):
                        data.append({
                            "장소": title.get_attribute('title'),
                            "도로명 주소": address.get_attribute('title'),
                            "지역": district
                        })
                    break
                
                # 페이지 번호 클릭 및 데이터 추출
                for page_num in range(1, 6):
                    try:
                        page_button = driver.find_element(By.CSS_SELECTOR, f"#info\\.search\\.page\\.no{page_num}")
                        page_button.click()
                        time.sleep(2)
                        
                        soup = bs(driver.page_source, "lxml")
                        
                        # 광고 제거
                        ads = soup.select("span.ico_ad")
                        for ad in ads:
                            ad.extract()
                        
                        # 장소 이름과 도로명 주소 추출
                        titles = soup.select("a.link_name")
                        addresses = soup.select("p[data-id='address']")
                        
                        for title, address in zip(titles, addresses):
                            data.append({
                                "장소": title['title'],
                                "도로명 주소": address['title'],
                                "지역": district
                            })
                    except:
                        continue
            
            except Exception as e:
                print(f"오류 발생: {e}")
                break
            
            # 다음 버튼 클릭
            try:
                next_button = driver.find_element(By.CSS_SELECTOR, "#info\\.search\\.page\\.next")
                if "disabled" in next_button.get_attribute("class"):
                    break
                next_button.click()
                time.sleep(2)
            except:
                break
    
    except Exception as e:
        print(f"오류 발생: {e}")

# 드라이버 닫기
driver.quit()

# 데이터를 DataFrame으로 변환
df = pd.DataFrame(data)

# DataFrame을 CSV 파일로 저장
df.to_csv("train_locations_different2.csv", index=False, encoding='utf-8-sig')

print("데이터가 'train_locations_different2.csv' 파일에 저장되었습니다.")


시외버스터미널

In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup as bs
import pandas as pd

driver = webdriver.Chrome()

# 검색할 구 목록
districts = ["경기도", "강원도", "경상북도", "경상남도", 
             "충청북도", "충청남도", "전라남도","전라북도",
             "제주도", "서울특별시", "부산광역시", "대구광역시",
             "인천광역시", "광주광역시", "대전광역시", "울산광역시",
             "세종특별자치시"
]


# 결과를 저장할 리스트 생성
data = []

for district in districts:
    # 카카오 지도 웹사이트 열기
    driver.get("https://map.kakao.com/")
    
    # 요소가 나타날 때까지 기다린 후 클릭
    time.sleep(2)
    try:
        img = driver.find_element(By.CSS_SELECTOR, "div.inner_coach_layer")
        img.click()
    except:
        pass
    
    # 검색 수행
    search = driver.find_element(By.CSS_SELECTOR, "#search\\.keyword\\.query")
    search.send_keys(f"시외버스터미널 {district}")
    search_btn = driver.find_element(By.CSS_SELECTOR, "#search\\.keyword\\.submit")
    search_btn.click()
    
    # 검색 결과가 로드될 시간을 줌
    time.sleep(2)
    
    # BeautifulSoup을 사용하여 페이지 소스 파싱
    soup = bs(driver.page_source, "lxml")
    
    # 광고 제거
    ads = soup.select("span.ico_ad")
    for ad in ads:
        ad.extract()
    
    # "장소 더보기" 버튼 클릭
    try:
        btn_more = driver.find_element(By.CSS_SELECTOR, "#info\\.search\\.place\\.more")
        btn_more.click()
        time.sleep(2)
    except:
        pass
    
    try:
        while True:
            # 1부터 5까지 페이지를 반복
            for page_num in range(1, 6):
                try:
                    page_button = driver.find_element(By.CSS_SELECTOR, f"#info\\.search\\.page\\.no{page_num}")
                    page_button.click()
                    time.sleep(2)
                    
                    soup = bs(driver.page_source, "lxml")
                    
                    # 광고 제거
                    ads = soup.select("span.ico_ad")
                    for ad in ads:
                        ad.extract()
                    
                    # 장소 이름과 도로명 주소 추출
                    titles = soup.select("a.link_name")
                    addresses = soup.select("p[data-id='address']")
                    
                    for title, address in zip(titles, addresses):
                        data.append({
                            "장소": title['title'],
                            "도로명 주소": address['title'],
                            "지역": district
                        })
                except:
                    break
            
            # 다음 버튼 클릭
            next_button = driver.find_element(By.CSS_SELECTOR, "#info\\.search\\.page\\.next")
            if "disabled" in next_button.get_attribute("class"):
                break
            next_button.click()
            time.sleep(2)
    except Exception as e:
        print(f"오류 발생: {e}")

# 드라이버 닫기
driver.quit()

# 데이터를 DataFrame으로 변환
df = pd.DataFrame(data)

# DataFrame을 CSV 파일로 저장
df.to_csv("bus_locations.csv", index=False, encoding='utf-8-sig')

print("데이터가 'bus_locations.csv' 파일에 저장되었습니다.")


In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
import time

# CSV 파일 경로
input_file_path = r'C:\Users\kfq\Downloads\1ST_Final_Project\크롤링\카카오맵 크롤링\bus_locations.csv'
output_file_path = r'C:\Users\kfq\Downloads\1ST_Final_Project\크롤링\카카오맵 크롤링\bus_locations_geocode.csv'

# CSV 파일 읽기
try:
    df = pd.read_csv(input_file_path, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(input_file_path, encoding='cp949')

# 지오코더 초기화
geolocator = Nominatim(user_agent="socar_geocoder", timeout=10)  # timeout 값을 10초로 설정

# 지오코딩 함수 정의
def geocode_address(address, retries=5):
    for attempt in range(retries):
        try:
            location = geolocator.geocode(address)
            if location:
                return location.latitude, location.longitude
        except GeocoderTimedOut:
            print(f"Geocoding timed out on attempt {attempt + 1}. Retrying...")
            time.sleep(2)  # 2초 대기 후 재시도
        except GeocoderServiceError as e:
            print(f"Geocoding service error: {e}")
        except Exception as e:
            print(f"Unexpected error: {e}")
    return None, None  # 재시도 후에도 실패하면 None 반환

# 위도와 경도 컬럼 추가
latitudes = []
longitudes = []

for index, row in df.iterrows():
    lat, lon = None, None
    if pd.notna(row['장소']):
        # 쏘카존 지점으로 지오코딩 시도
        lat, lon = geocode_address(row['장소'])
    if (lat is None or lon is None) and pd.notna(row['도로명 주소']):
        # 실패하면 도로명 주소로 지오코딩 시도
        lat, lon = geocode_address(row['도로명 주소'])
    
    latitudes.append(lat)
    longitudes.append(lon)

df['위도'] = latitudes
df['경도'] = longitudes

# 위도와 경도만 포함된 새로운 데이터프레임 생성
df_geocoded = df[['위도', '경도']]

# 결과를 새로운 CSV 파일로 저장 (utf-8 인코딩)
df_geocoded.to_csv(output_file_path, index=False, encoding='utf-8')

print("지오코딩이 완료되었습니다. 결과가 저장되었습니다:", output_file_path)


고속버스터미널

In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup as bs
import pandas as pd

# webdriver_manager를 사용하여 Chrome 드라이버 설정

driver = webdriver.Chrome()

# 검색할 구 목록
districts = ["경기도", "강원도", "경상북도", "경상남도", 
             "충청북도", "충청남도", "전라남도","전라북도",
             "제주도", "서울특별시", "부산광역시", "대구광역시",
             "인천광역시", "광주광역시", "대전광역시", "울산광역시",
             "세종특별자치시"
]


# 결과를 저장할 리스트 생성
data = []

for district in districts:
    # 카카오 지도 웹사이트 열기
    driver.get("https://map.kakao.com/")
    
    # 요소가 나타날 때까지 기다린 후 클릭
    time.sleep(2)
    try:
        img = driver.find_element(By.CSS_SELECTOR, "div.inner_coach_layer")
        img.click()
    except:
        pass
    
    # 검색 수행
    search = driver.find_element(By.CSS_SELECTOR, "#search\\.keyword\\.query")
    search.send_keys(f"고속버스터미널 {district}")
    search_btn = driver.find_element(By.CSS_SELECTOR, "#search\\.keyword\\.submit")
    search_btn.click()
    
    # 검색 결과가 로드될 시간을 줌
    time.sleep(2)
    
    # BeautifulSoup을 사용하여 페이지 소스 파싱
    soup = bs(driver.page_source, "lxml")
    
    # 광고 제거
    ads = soup.select("span.ico_ad")
    for ad in ads:
        ad.extract()
    
    # "장소 더보기" 버튼 클릭
    try:
        btn_more = driver.find_element(By.CSS_SELECTOR, "#info\\.search\\.place\\.more")
        btn_more.click()
        time.sleep(2)
    except:
        pass
    
    try:
        while True:
            # 1부터 5까지 페이지를 반복
            for page_num in range(1, 6):
                try:
                    page_button = driver.find_element(By.CSS_SELECTOR, f"#info\\.search\\.page\\.no{page_num}")
                    page_button.click()
                    time.sleep(2)
                    
                    soup = bs(driver.page_source, "lxml")
                    
                    # 광고 제거
                    ads = soup.select("span.ico_ad")
                    for ad in ads:
                        ad.extract()
                    
                    # 장소 이름과 도로명 주소 추출
                    titles = soup.select("a.link_name")
                    addresses = soup.select("p[data-id='address']")
                    
                    for title, address in zip(titles, addresses):
                        data.append({
                            "장소": title['title'],
                            "도로명 주소": address['title'],
                            "지역": district
                        })
                except:
                    break
            
            # 다음 버튼 클릭
            next_button = driver.find_element(By.CSS_SELECTOR, "#info\\.search\\.page\\.next")
            if "disabled" in next_button.get_attribute("class"):
                break
            next_button.click()
            time.sleep(2)
    except Exception as e:
        print(f"오류 발생: {e}")

# 드라이버 닫기
driver.quit()

# 데이터를 DataFrame으로 변환
df = pd.DataFrame(data)

# DataFrame을 CSV 파일로 저장
df.to_csv("gosok2.csv", index=False, encoding='utf-8-sig')

print("데이터가 'gosok2.csv' 파일에 저장되었습니다.")
